In [4]:
# 09 — HCAD Data Preparation (OSM + HCAD Consolidation)
# Fetches OSM buildings, spatially joins to HCAD, consolidates into single CSV

import pandas as pd
import numpy as np
import requests
import json
import os
import hashlib
import time
from sklearn.neighbors import BallTree

# ── PARAMETERS ──────────────────────────────────────
HOUSTON_BBOX = [29.62, -95.82, 29.85, -95.19]  # [south, west, north, east]
CACHE_DIR = "houston_data/cache"
OUTPUT_DIR = "hcad"
SEARCH_RADIUS_M = 50  # Search for OSM buildings within 50m
MAX_RETRIES = 3

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("09 — HCAD Data Preparation (OSM + HCAD Consolidation)")
print(f"Houston bbox: {HOUSTON_BBOX}")
print()

# ── STEP 1: Fetch OSM Buildings (with retry) ─────────
print("Step 1: Fetching OSM buildings from Overpass API...")

def fetch_osm_buildings(bbox, cache_dir, max_retries=3):
    """Fetch buildings from OSM Overpass API with caching and retry logic."""
    query_hash = hashlib.sha1(json.dumps(bbox).encode()).hexdigest()
    cache_file = os.path.join(cache_dir, f"osm_buildings_{query_hash}.json")
    
    if os.path.exists(cache_file):
        print(f"  Loading from cache: {cache_file}")
        with open(cache_file, "r") as f:
            return json.load(f)
    
    print(f"  Querying Overpass API (this may take 5-10 minutes)...")
    
    s, w, n, e = bbox
    overpass_query = f"""[bbox:{s},{w},{n},{e}];(way["building"];relation["building"];);out center;"""
    
    # Multiple Overpass endpoints to try
    urls = [
        "https://lz4.overpass-api.de/api/interpreter",
        "https://overpass.kumi.systems/api/interpreter",
        "https://overpass.openstreetmap.ru/cgi/interpreter",
        "http://overpass-api.de/api/interpreter"
    ]
    
    for attempt in range(max_retries):
        for url in urls:
            try:
                endpoint = url.split('/')[2]
                print(f"  Attempt {attempt+1}/{max_retries}, trying {endpoint}...")
                response = requests.post(url, data=overpass_query, timeout=600)
                response.raise_for_status()
                data = response.json()
                
                # Cache the result
                with open(cache_file, "w") as f:
                    json.dump(data, f)
                print(f"  ✓ Success! Cached: {cache_file}")
                return data
            except requests.exceptions.Timeout:
                print(f"    Timeout (waiting for slower endpoint...)")
            except Exception as e:
                print(f"    Failed: {type(e).__name__}")
        
        # Exponential backoff between attempts
        if attempt < max_retries - 1:
            wait_time = 2 ** (attempt + 1)
            print(f"  Waiting {wait_time}s before retry...")
            time.sleep(wait_time)
    
    print(f"  All {max_retries} attempts exhausted.")
    return None

osm_data = fetch_osm_buildings(HOUSTON_BBOX, CACHE_DIR, MAX_RETRIES)

if osm_data is None:
    print("  Failed to fetch OSM data after all retries.")
    print("  Using coordinate/floor defaults from HCAD only.")
    osm_buildings = pd.DataFrame(columns=["lat", "lon", "building_levels"])
else:
    # Parse OSM data
    osm_buildings = []
    for element in osm_data.get("elements", []):
        if "center" in element:
            lat = element["center"]["lat"]
            lon = element["center"]["lon"]
            levels = element.get("tags", {}).get("building:levels", None)
            osm_buildings.append({
                "lat": lat,
                "lon": lon,
                "building_levels": levels
            })
    
    osm_buildings = pd.DataFrame(osm_buildings)
    
    if len(osm_buildings) > 0:
        print(f"  ✓ Downloaded {len(osm_buildings)} buildings from OSM")
        osm_buildings["building_levels"] = pd.to_numeric(
            osm_buildings["building_levels"], errors="coerce"
        ).fillna(1).astype(int)
    else:
        print(f"  No buildings found in response (empty result).")
        osm_buildings = pd.DataFrame(columns=["lat", "lon", "building_levels"])

print()

# ── STEP 2: Load & Aggregate HCAD Data ──────────────
print("Step 2: Loading HCAD data...")

csv_base = "Houston_data/csv_files/Real_building_land"

# Load HCAD tables
land = pd.read_csv(f"{csv_base}/land.csv", dtype={"use_cd": str})
exterior = pd.read_csv(f"{csv_base}/exterior.csv")
extra_features_detail1 = pd.read_csv(f"{csv_base}/extra_features_detail1.csv")
structural_elem1 = pd.read_csv(f"{csv_base}/structural_elem1.csv")

print(f"  land.csv: {len(land)} rows")
print(f"  exterior.csv: {len(exterior)} rows")
print(f"  extra_features_detail1.csv: {len(extra_features_detail1)} rows")
print(f"  structural_elem1.csv: {len(structural_elem1)} rows")

# Aggregate by account
print("  Aggregating by account...")

# From land: lotarea, landuse
land_agg = land.groupby("acct").agg({
    "uts": "sum",  # total lot square footage
    "use_cd": lambda x: x.iloc[0]  # take first land use code
}).rename(columns={"uts": "lotarea", "use_cd": "landuse"}).reset_index()

# From exterior: building area (sum of all exterior areas)
exterior_agg = exterior.groupby("acct").agg({
    "area": "sum"  # total exterior area
}).rename(columns={"area": "bldgarea"}).reset_index()

# From extra_features_detail1: year built (take most recent)
extra_features_agg = extra_features_detail1.groupby("acct").agg({
    "act_yr": "max"  # use most recent year built
}).rename(columns={"act_yr": "yearbuilt"}).reset_index()

# From structural_elem1: number of buildings (count distinct bld_num)
struct_agg = structural_elem1.groupby("acct").agg({
    "bld_num": "nunique"  # count distinct buildings
}).rename(columns={"bld_num": "numbldgs"}).reset_index()

# Merge all
hcad_agg = land_agg.copy()
for df in [exterior_agg, extra_features_agg, struct_agg]:
    hcad_agg = hcad_agg.merge(df, on="acct", how="left")

# Fill missing values
hcad_agg["yearbuilt"] = pd.to_numeric(hcad_agg["yearbuilt"], errors="coerce").fillna(0).astype(int)
hcad_agg["numbldgs"] = hcad_agg["numbldgs"].fillna(1).astype(int)
hcad_agg["bldgarea"] = hcad_agg["bldgarea"].fillna(0)
hcad_agg["lotarea"] = hcad_agg["lotarea"].fillna(0)

print(f"  ✓ Aggregated to {len(hcad_agg)} unique accounts")
print()

# ── STEP 3: Calculate Account Centroids ──────────────
print("Step 3: Assigning coordinates...")

center_lat = (HOUSTON_BBOX[0] + HOUSTON_BBOX[2]) / 2
center_lon = (HOUSTON_BBOX[1] + HOUSTON_BBOX[3]) / 2

hcad_agg["lat_approx"] = center_lat
hcad_agg["lon_approx"] = center_lon

print(f"  Using bbox center: ({center_lat:.4f}, {center_lon:.4f})")
print()

# ── STEP 4: Spatial Join - HCAD to nearest OSM building ─
print("Step 4: Matching building characteristics...")

if len(osm_buildings) == 0:
    print("  No OSM data available → using defaults")
    print("    • latitude/longitude: bbox center")
    print("    • numfloors: 1 (conservative estimate)")
    hcad_agg["latitude"] = hcad_agg["lat_approx"]
    hcad_agg["longitude"] = hcad_agg["lon_approx"]
    hcad_agg["numfloors"] = 1
else:
    # Build BallTree for fast nearest-neighbor search
    osm_coords = np.radians(osm_buildings[["lat", "lon"]].values)
    tree = BallTree(osm_coords, metric="haversine")
    
    hcad_coords = np.radians(hcad_agg[["lat_approx", "lon_approx"]].values)
    
    # Find nearest OSM building for each HCAD property
    distances, indices = tree.query(hcad_coords, k=1)
    
    # Convert distances from radians to meters (Earth radius ~ 6371 km)
    distances_m = distances.flatten() * 6371000
    
    # Extract OSM building data for nearest neighbors
    nearest_osm = osm_buildings.iloc[indices.flatten()].reset_index(drop=True)
    nearest_osm["distance_m"] = distances_m
    
    # Use nearest OSM building if within search radius
    use_osm = distances_m <= SEARCH_RADIUS_M
    
    hcad_agg["latitude"] = np.where(
        use_osm,
        nearest_osm["lat"],
        hcad_agg["lat_approx"]
    )
    hcad_agg["longitude"] = np.where(
        use_osm,
        nearest_osm["lon"],
        hcad_agg["lon_approx"]
    )
    hcad_agg["numfloors"] = np.where(
        use_osm,
        nearest_osm["building_levels"],
        1  # Default to 1 floor if no OSM match
    )
    
    osm_matched = use_osm.sum()
    print(f"  ✓ Matched {osm_matched:,}/{len(hcad_agg):,} properties to OSM buildings (within {SEARCH_RADIUS_M}m)")

print()

# ── STEP 5: Consolidate Final Output ─────────────────
print("Step 5: Consolidating final output...")

final_df = hcad_agg[[
    "acct", "latitude", "longitude", "numfloors", "yearbuilt",
    "bldgarea", "numbldgs", "lotarea", "landuse"
]].copy()

# Ensure correct data types
final_df["latitude"] = final_df["latitude"].astype(float)
final_df["longitude"] = final_df["longitude"].astype(float)
final_df["numfloors"] = final_df["numfloors"].astype(int)
final_df["yearbuilt"] = final_df["yearbuilt"].astype(int)
final_df["bldgarea"] = final_df["bldgarea"].astype(float)
final_df["numbldgs"] = final_df["numbldgs"].astype(int)
final_df["lotarea"] = final_df["lotarea"].astype(float)
final_df["landuse"] = final_df["landuse"].astype(str)

output_path = f"{OUTPUT_DIR}/harris_county_building_data.csv"
final_df.to_csv(output_path, index=False, encoding="utf-8")

print(f"  ✓ Saved {len(final_df):,} properties to {output_path}")
print()

# ── SUMMARY ──────────────────────────────────────────
print("="*60)
print("  CONSOLIDATION COMPLETE")
print("="*60)
print(f"Output: {output_path}")
print(f"Records: {len(final_df):,}")
print(f"Columns: {', '.join(final_df.columns.tolist())}")
print()
print("Sample rows:")
print(final_df.head())
print()

09 — HCAD Data Preparation (OSM + HCAD Consolidation)
Houston bbox: [29.62, -95.82, 29.85, -95.19]

Step 1: Fetching OSM buildings from Overpass API...
  Querying Overpass API (this may take 5-10 minutes)...
  Attempt 1/3, trying lz4.overpass-api.de...
    Failed: HTTPError
  Attempt 1/3, trying overpass.kumi.systems...
    Failed: HTTPError
  Attempt 1/3, trying overpass.openstreetmap.ru...
    Timeout (waiting for slower endpoint...)
  Attempt 1/3, trying overpass-api.de...
    Failed: HTTPError
  Waiting 2s before retry...
  Attempt 2/3, trying lz4.overpass-api.de...
    Failed: HTTPError
  Attempt 2/3, trying overpass.kumi.systems...
    Failed: HTTPError
  Attempt 2/3, trying overpass.openstreetmap.ru...
    Timeout (waiting for slower endpoint...)
  Attempt 2/3, trying overpass-api.de...
    Failed: HTTPError
  Waiting 4s before retry...
  Attempt 3/3, trying lz4.overpass-api.de...
    Failed: HTTPError
  Attempt 3/3, trying overpass.kumi.systems...
    Failed: JSONDecodeError
  